In [1]:
import torch
import torch.nn as nn

# ──────────────────────────────────────────────
# 1. Manual bias implementation
# ──────────────────────────────────────────────

class LinearNoBias(nn.Module):
    """Linear layer WITHOUT bias — manual matmul only."""

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # y = xW^T  (no bias term)
        return x @ self.weight.t()


class LinearWithBias(nn.Module):
    """Linear layer WITH manually added bias."""

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias   = nn.Parameter(torch.zeros(out_features))  # ← the bias

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # y = xW^T + b
        return x @ self.weight.t() + self.bias   # ← bias shifts the output


# ──────────────────────────────────────────────
# 2. What bias actually does (visual explanation)
# ──────────────────────────────────────────────

def show_bias_effect():
    """Demonstrate how bias shifts the output of a neuron."""

    torch.manual_seed(42)
    x = torch.tensor([[1.0, 2.0]])       # single input sample
    w = torch.tensor([[0.5, -0.3]])       # single neuron weights

    no_bias  = x @ w.t()                  # without bias
    bias     = torch.tensor([0.7])
    with_bias = x @ w.t() + bias          # with bias

    print("── What bias does (single neuron) ──\n")
    print(f"  Input x:        {x.squeeze().tolist()}")
    print(f"  Weights w:      {w.squeeze().tolist()}")
    print(f"  Bias b:         {bias.item()}\n")
    print(f"  Without bias:   x·w = {no_bias.item():+.2f}")
    print(f"  With bias:      x·w + b = {with_bias.item():+.2f}")
    print(f"  Difference:     {bias.item():+.2f}  (bias shifts the decision boundary)\n")

    # Through sigmoid
    print("  After sigmoid:")
    print(f"    σ(x·w)     = {torch.sigmoid(no_bias).item():.4f}")
    print(f"    σ(x·w + b) = {torch.sigmoid(with_bias).item():.4f}")
    print(f"    → Bias shifted the probability by "
          f"{torch.sigmoid(with_bias).item() - torch.sigmoid(no_bias).item():+.4f}\n")


# ──────────────────────────────────────────────
# 3. Training comparison: bias vs no bias
# ──────────────────────────────────────────────

def train_and_compare():
    """Train two models on data that NEEDS a bias to be learned well."""

    torch.manual_seed(0)

    # Dataset: y = 1 when x1 + x2 > 3  (decision boundary NOT through origin)
    # A model without bias can only learn boundaries through the origin,
    # so it will struggle here.
    X = torch.randn(200, 2) * 2 + 2     # centered around (2, 2)
    y = ((X[:, 0] + X[:, 1]) > 3).float().unsqueeze(1)

    print("── Training comparison ──\n")
    print(f"  Dataset: 200 samples, boundary at x1 + x2 = 3")
    print(f"  Class balance: {y.mean().item():.0%} positive\n")

    results = {}

    for label, LayerClass in [("No Bias ", LinearNoBias),
                               ("With Bias", LinearWithBias)]:

        torch.manual_seed(42)

        model = nn.Sequential(
            LayerClass(2, 8),
            nn.ReLU(),
            LayerClass(8, 1),
            nn.Sigmoid(),
        )

        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        # Train
        for epoch in range(1, 501):
            pred = model(X)
            loss = criterion(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Evaluate
        with torch.no_grad():
            acc = ((model(X) >= 0.5).float() == y).float().mean().item()

        results[label] = {"loss": loss.item(), "accuracy": acc}
        print(f"  {label}  →  Loss: {loss.item():.4f}  |  Accuracy: {acc:.2%}")

    gap = results["With Bias"]["accuracy"] - results["No Bias "]["accuracy"]
    print(f"\n  Accuracy gap: {gap:+.2%} in favor of bias")
    print("  → Bias lets the model shift decision boundaries away from the origin.\n")


# ──────────────────────────────────────────────
# 4. Verify manual bias matches nn.Linear
# ──────────────────────────────────────────────

def verify_against_pytorch():
    """Prove our manual bias gives the same result as nn.Linear."""

    torch.manual_seed(7)
    x = torch.randn(3, 4)  # batch of 3, 4 features

    # PyTorch built-in
    builtin = nn.Linear(4, 2, bias=True)

    # Our manual version with same weights
    manual = LinearWithBias(4, 2)
    manual.weight.data = builtin.weight.data.clone()
    manual.bias.data   = builtin.bias.data.clone()

    out_builtin = builtin(x)
    out_manual  = manual(x)

    match = torch.allclose(out_builtin, out_manual, atol=1e-6)

    print("── Verification: manual bias == nn.Linear ──\n")
    print(f"  nn.Linear output:\n{out_builtin}\n")
    print(f"  Manual output:\n{out_manual}\n")
    print(f"  Match: {match}\n")


# ──────────────────────────────────────────────
# 5. Run everything
# ──────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 55)
    print("  Manual Bias Addition & Its Effect")
    print("=" * 55 + "\n")

    show_bias_effect()
    train_and_compare()
    verify_against_pytorch()

    print("── Key takeaway ──")
    print("  Without bias: decision boundary must pass through the origin.")
    print("  With bias:    boundary can shift anywhere → much more flexible.")

  Manual Bias Addition & Its Effect

── What bias does (single neuron) ──

  Input x:        [1.0, 2.0]
  Weights w:      [0.5, -0.30000001192092896]
  Bias b:         0.699999988079071

  Without bias:   x·w = -0.10
  With bias:      x·w + b = +0.60
  Difference:     +0.70  (bias shifts the decision boundary)

  After sigmoid:
    σ(x·w)     = 0.4750
    σ(x·w + b) = 0.6457
    → Bias shifted the probability by +0.1706

── Training comparison ──

  Dataset: 200 samples, boundary at x1 + x2 = 3
  Class balance: 70% positive

  No Bias   →  Loss: 0.2751  |  Accuracy: 84.00%
  With Bias  →  Loss: 0.0165  |  Accuracy: 100.00%

  Accuracy gap: +16.00% in favor of bias
  → Bias lets the model shift decision boundaries away from the origin.

── Verification: manual bias == nn.Linear ──

  nn.Linear output:
tensor([[ 0.4131, -0.1287],
        [ 0.1021,  0.9492],
        [ 0.1104,  0.8923]], grad_fn=<AddmmBackward0>)

  Manual output:
tensor([[ 0.4131, -0.1287],
        [ 0.1021,  0.9492],
   